# 29 · Why a gateway: one endpoint, many replicas
Level L5 · Part 5 · core · ~15 min · builds on 15, 28

Time: about 15 min to read and run. This lesson is short, but it is still worth running for the gateway numbers. The live run starts three **simulated** replicas and a LiteLLM proxy on this machine: no Docker, no GPU, no key. Without them, the tables come from the **recorded output** of a live run (`data/recorded/29.json`), labelled as such.

A **replica** is a copy of the same model server, on its own GPU or server (here, a simulated copy). A **gateway** is the one endpoint that takes every request and sends each one to a replica.

Ports: 8290 (the gateway), 8291 to 8293 (the replicas). Replica b is listed but not started: it is down.

## Goal
By the end you can show, with numbers, what goes wrong when each app picks a replica itself (failed requests, load that ignores how busy each replica is) and what changes when the same traffic goes through one gateway URL that routes, retries and cools down a bad replica.

## Why it matters
- **Without it:** each app keeps the replica list, picks one and handles its own failures. Ten apps and four replicas means forty links to configure and ten copies of the list. Each app sees only its own traffic, so none of them knows that one replica is busy or down.
- **With it:** apps call one URL under one model name. The gateway holds the list, sends each request to the least busy replica, retries a failed call and stops sending to a replica that keeps failing. An app changes one line: its URL.
- **When you don't need it:** one replica, a laptop, or a batch job that can wait. A gateway is one more hop and one more thing to run.

## The idea
A gateway is the single entry point in front of several identical replicas. It does three jobs that apps should not each do: it chooses a replica using a view of *all* the traffic, it retries and cools down replicas that fail, and it hides the list, so replicas can be added or removed without touching any app. Think of a hotel front desk: guests do not walk to the rooms and pick one; the desk knows which rooms are free. The gateway does not make a model faster. It decides where each request goes.

## Picture
Without a gateway, every app holds the list. The apps do not see each other's traffic.
```mermaid
flowchart LR
  A1[App 1 with its own list] --> R1[Replica a]
  A1 --> R2[Replica b: down]
  A1 --> R3[Replica c: slow]
  A2[App 2 with its own list] --> R1
  A2 --> R2
  A2 --> R3
```
With a gateway, the apps see one URL and the gateway sees all the traffic.
```mermaid
flowchart LR
  A1[App 1] --> G[Gateway: one URL, least busy, retries]
  A2[App 2] --> G
  G --> R1[Replica a]
  G -.->|cooled down| R2[Replica b: down]
  G --> R3[Replica c: slow]
```

## Already know this?
This lesson opens Part 5 (serving at scale). Answer these five from lessons 15, 27 and 28. If you can answer all five, skip to lesson 30.
1. Name three of the five questions that decide which serving engine fits a workload. (15)
2. Why must a load test send requests on its own clock, not wait for each answer? (27)
3. Why does p95 tell you more than the mean about what a user feels? (27)
4. What happens to a request that arrives when all decode slots are busy? (27)
5. Which three inputs turn tokens per second into cost per million tokens? (28)

## Step by step

### Step 1: the replicas, and what is simulated
A replica here is a *simulated* server: an OpenAI-style chat endpoint with four decode slots, like the lab server in lesson 27. It waits `max_tokens` times a speed setting, then answers with a text that says no model ran. Real vLLM replicas took about ten minutes to start on this 4-core machine (lesson 27 measured 604 s and 618 s), and two of them would fight for the same cores. So the timing is what is under test, and no answer is model output. Replica c is twice as slow as a and b (40 ms per token against 20).

In [1]:
from pathlib import Path
Path("../apps/29").mkdir(parents=True, exist_ok=True); Path("../data/29").mkdir(parents=True, exist_ok=True)
print("folders ready:", Path("../apps/29").is_dir(), Path("../data/29").is_dir())

folders ready: True True


The replica file is written next. The folders above exist first, so a fresh checkout runs the cell.

In [2]:
%%writefile ../apps/29/replica.py
"""SIMULATED replica for lesson 29: an OpenAI-style chat endpoint that takes time but runs no model.
It has SLOTS decode slots like the lab server; an answer takes max_tokens x MS_PER_TOKEN milliseconds."""
import asyncio, sys, time
from fastapi import FastAPI, Request
import uvicorn

NAME, PORT, MS_PER_TOKEN, SLOTS = sys.argv[1], int(sys.argv[2]), float(sys.argv[3]), int(sys.argv[4])
app = FastAPI()
slots = asyncio.Semaphore(SLOTS)
state = {"served": 0, "in_flight": 0}

@app.post("/v1/chat/completions")
async def chat(req: Request):
    body = await req.json()
    n = int(body.get("max_tokens", 48))
    state["in_flight"] += 1
    try:
        async with slots:                                  # a request waits here while all slots are busy
            await asyncio.sleep(n * MS_PER_TOKEN / 1000)
            state["served"] += 1
    finally:
        state["in_flight"] -= 1
    return {"id": "sim", "object": "chat.completion", "created": int(time.time()), "model": body.get("model"),
            "choices": [{"index": 0, "finish_reason": "length", "message": {"role": "assistant",
                         "content": f"[simulated replica {NAME}: no model ran]"}}],
            "usage": {"prompt_tokens": 0, "completion_tokens": n, "total_tokens": n}}

@app.get("/stats")
def stats():
    return {"replica": NAME, **state}

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

Overwriting ../apps/29/replica.py


### Step 2: the world without a gateway
Each app keeps the replica list and picks one per request. The simplest rule is a random choice, and it is what many apps do first. The helpers below are the client side: the same 120 requests, 24 in flight, with lengths between 24 and 72 tokens. The traffic is made up in the cell (a fixed seed, no data file, no licence to check). The `pick` function is the only thing that differs between the two runs later.

In [3]:
import asyncio, atexit, json, os, random, secrets, subprocess, sys, time
import httpx

REPLICA_PORTS = {"a": 8291, "b": 8292, "c": 8293}          # b is in every app's list, but it is down
SPEED = {"a": 20, "b": 20, "c": 40}                        # ms per answer token: c is the slow one
GATEWAY = "http://127.0.0.1:8290/v1/chat/completions"      # Step 5
KEY = secrets.token_hex(16)                                # a gateway key made fresh in each run: never printed or saved
AUTH = {"Authorization": f"Bearer {KEY}"}
RESULTS, PROCS, NO_KEY, RECORDED = {}, [], {}, set()
REC = json.loads(Path("../data/recorded/29.json").read_text())   # the recorded run, used for any row not run live

def pick_without(rng):
    """What each app does today: keep the replica list and pick one at random, per request."""
    return f"http://127.0.0.1:{rng.choice(list(REPLICA_PORTS.values()))}/v1/chat/completions"

The next helper counts what each replica has served, and waits until a URL answers. Both read the replicas directly, so the counts do not depend on the apps.

In [4]:
def served():
    """Answers each replica has sent so far; None when a replica is down."""
    out = {}
    for name, port in REPLICA_PORTS.items():
        try: out[name] = httpx.get(f"http://127.0.0.1:{port}/stats", timeout=2).json()["served"]
        except httpx.TransportError: out[name] = None
    return out

def wait_http(url, limit=60):
    """Poll a URL until it answers (any status) or the limit passes."""
    end = time.time() + limit
    while time.time() < end:
        try: httpx.get(url, timeout=2); return
        except httpx.TransportError: time.sleep(0.5)
    raise TimeoutError(f"no answer from {url} after {limit} s")

The client: 120 requests, 24 in flight. `pick(rng)` gives the URL of each request. In Step 2 it is a replica chosen at random; in Step 6 it is the gateway. Nothing else changes.

In [5]:
async def load(label, pick, n=120, in_flight=24):
    """n requests from the apps, `in_flight` at a time; the result goes into RESULTS[label]."""
    rng = random.Random(29); gate = asyncio.Semaphore(in_flight); lat, errors = [], 0
    urls = [pick(rng) for _ in range(n)]; lengths = [rng.randint(24, 72) for _ in range(n)]
    before = served()
    async with httpx.AsyncClient(timeout=120, headers=AUTH) as client:
        async def one(url, max_tokens):
            nonlocal errors
            async with gate:
                t0 = time.perf_counter()
                try:
                    r = await client.post(url, json={"model": "brightside-qwen", "max_tokens": max_tokens,
                                                      "messages": [{"role": "user", "content": "refund policy?"}]})
                    r.raise_for_status(); lat.append(time.perf_counter() - t0)
                except httpx.HTTPError:
                    errors += 1
        await asyncio.gather(*(one(u, t) for u, t in zip(urls, lengths)))
    after = served()
    done = sorted(lat); p95 = done[int(0.95 * len(done)) - 1]
    got = {k: (after[k] - before[k] if None not in (before[k], after[k]) else None) for k in after}
    RESULTS[label] = {"ok": len(lat), "errors": errors, "served": got, "p95_s": round(p95, 1)}
    print(f"{label}: {len(lat)} answered, {errors} failed")

Two helpers turn the results into one row per run. `fill_missing` loads a row from the recording for any label that was not run live, and marks it as recorded, so a table is never half live and half blank.

In [6]:
def fill_missing(labels=None):
    """Fill each missing label from the recording and say so. Without labels, fill every row."""
    for k in (labels or REC["results"]):
        if k not in RESULTS:
            RESULTS[k] = REC["results"][k]; RECORDED.add(k)
            print(f"source: recorded output, row '{k}'")

def show(labels=None):
    """One row per run: successes, failures, what each replica served, and the p95 wait of the successes."""
    for k, r in RESULTS.items():
        if labels and k not in labels: continue
        s = " ".join(f"{n} {'down' if v is None else v:>4}" for n, v in r["served"].items())
        tag = "  (recorded)" if k in RECORDED else ""
        print(f"{k:<20} ok {r['ok']:>4}  errors {r['errors']:>3}  served: {s}  p95 {r['p95_s']:>5} s{tag}")

### Step 3: start the replicas and run the apps' rule
**To run this live:** set `TUT_ENABLE=slow` before you start Jupyter (or in the command that runs the notebook). Otherwise the slow steps are skipped and the tables show the recording. Run all the `slow` cells or none: a table with some rows live and some recorded is not a fair comparison.

Start replicas a and c. Replica b stays down, as it would after a crash that nobody has noticed yet. Then send the 120 requests with the random rule.

In [7]:
def start_replicas():
    for name, port in REPLICA_PORTS.items():
        if name != "b":
            PROCS.append(subprocess.Popen([sys.executable, "../apps/29/replica.py", name, str(port), str(SPEED[name]), "4"]))
    for name, port in REPLICA_PORTS.items():
        if name != "b": wait_http(f"http://127.0.0.1:{port}/stats")
    print("replicas a and c answer; b is down")
atexit.register(lambda: [p.terminate() for p in PROCS])
start_replicas()

⏭  Skipped: this cell needs [slow]. Run it yourself, or re-run with TUT_ENABLE=slow.


Now the apps send their 120 requests, each to a replica the app picked. Replica b refuses every call.

In [8]:
await load("without: apps pick", pick_without)
show()

⏭  Skipped: this cell needs [slow]. Run it yourself, or re-run with TUT_ENABLE=slow.


### Step 4: measure the imbalance
The table has one row per run; this step shows the run without a gateway. `errors` counts requests the app saw fail. `served` counts what each replica answered, read from the replica itself. `p95` is the wait of the successful requests at the 95th percentile.

**Read it.** Of 120 requests, 40 failed: the random rule sent a third of the traffic to b, and b refused every call. Replica c answered about as many requests as a (42 against 38) although each of its answers takes twice as long, and 52% of the successful requests went to c. Nothing in the apps can see that, because no app knows what the others send. (These are the numbers printed in Step 3, or the recorded run if Step 3 was skipped.)

In [9]:
fill_missing(["without: apps pick"])
show(["without: apps pick"])

source: recorded output, row 'without: apps pick'
without: apps pick   ok   80  errors  40  served: a   38 b down c   42  p95  10.3 s  (recorded)


### Step 5: put one gateway in front
The gateway is a LiteLLM proxy, an OpenAI-compatible server that gives one URL in front of model deployments. Every entry below has the same `model_name`, so the proxy treats the three replicas as one model. The router settings ask for least-busy routing (the deployment with the fewest calls in flight gets the next one), two retries, and a cooldown. `allowed_fails: 1` means a replica may fail once a minute; a replica that fails more than that is cooled down, and `cooldown_time: 30` keeps it out of the rotation for 30 s.

Why least-busy helps: a faster replica finishes its calls sooner, so it has fewer calls in flight and the router picks it more often. Step 7 shows that ratio.

In [10]:
%%writefile ../data/29/config.yaml
model_list:
  - model_name: brightside-qwen
    litellm_params:
      model: openai/brightside-qwen-0.5b
      api_base: http://127.0.0.1:8291/v1
      api_key: none
  - model_name: brightside-qwen
    litellm_params:
      model: openai/brightside-qwen-0.5b
      api_base: http://127.0.0.1:8292/v1
      api_key: none
  - model_name: brightside-qwen
    litellm_params:
      model: openai/brightside-qwen-0.5b
      api_base: http://127.0.0.1:8293/v1
      api_key: none
router_settings:
  routing_strategy: least-busy
  num_retries: 2
  allowed_fails: 1
  cooldown_time: 30

Overwriting ../data/29/config.yaml


The proxy refuses to start without a master key. The cell gives it the key from this run through its environment: `KEY` is made fresh each time the notebook runs and is never printed or saved. In a real deployment the key lives in `.env`. Start the proxy, then wait until it answers a real request.

In [11]:
def start_gateway(limit=120):
    env = {**os.environ, "LITELLM_MASTER_KEY": KEY}        # the proxy reads its master key from the environment
    litellm = Path(sys.executable).parent / "litellm"
    PROCS.append(subprocess.Popen([str(litellm), "--config", "config.yaml", "--port", "8290"], cwd="../data/29", env=env,
                                  stdout=open("../data/29/proxy.log", "w"), stderr=subprocess.STDOUT))
    end = time.time() + limit
    while time.time() < end:
        try:
            ask = {"model": "brightside-qwen", "max_tokens": 1, "messages": [{"role": "user", "content": "ready?"}]}
            if httpx.post(GATEWAY, headers=AUTH, json=ask, timeout=60).status_code == 200: return
        except httpx.TransportError: pass
        time.sleep(1)
    raise TimeoutError("the gateway did not answer: see ../data/29/proxy.log")
start_gateway()

⏭  Skipped: this cell needs [slow]. Run it yourself, or re-run with TUT_ENABLE=slow.


### Step 6: the same traffic through the gateway
The same 120 requests, the same replicas, the same lengths. The only change is the `pick` function: every request now goes to one URL. Nothing else in the apps changes.

In [12]:
await load("with: one gateway", lambda _rng: GATEWAY)
show()

⏭  Skipped: this cell needs [slow]. Run it yourself, or re-run with TUT_ENABLE=slow.


### Step 7: read the two runs
Both runs go into one table. Any row that was not run live is filled from the recording and marked "(recorded)".

In [13]:
fill_missing()
show()

source: recorded output, row 'with: one gateway'
without: apps pick   ok   80  errors  40  served: a   38 b down c   42  p95  10.3 s  (recorded)
with: one gateway    ok  120  errors   0  served: a   80 b down c   40  p95   7.0 s  (recorded)


**Read it.** All 120 requests were answered; none failed for the client. Replica b answered none, because it is down. The table does not count retries or cooldowns, and this lesson does not print them: the proxy log is where they would show. The served counts show the routing: a, the faster replica, answers about twice as many requests as c, so c's share of the successes falls from about half to about a third. The p95 wait falls from about 10 s to about 7 s. The gateway does not make c faster; it sends c less work, because c usually has calls in flight.

## How it simplifies the architecture
The counts below are for the picture (2 apps, 3 replicas) and for a 10-app, 4-replica fleet.

| | Without a gateway | With a gateway |
| --- | --- | --- |
| Links to configure | apps × replicas: 2 × 3 = 6; 10 × 4 = 40 | apps + replicas: 2 + 3 = 5; 10 + 4 = 14 |
| Copies of the replica list | one in every app | one, in the gateway |
| Retry and cooldown rules | one in every app, often none | one, in the gateway |
| Who sees the load | each app, its own share only | the gateway, all traffic |
| New failure point | none: each app fails alone | the gateway: run two and watch its health |

**What the gateway does not solve.** It cannot shorten a replica's queue: when all slots are busy, requests wait. It does not know which prompts share a prefix cache; cache-aware routing needs a router that does (lesson 30). It is one component: run two, or it becomes the outage.

**When you don't need it:** one replica. **Alternatives:** the vLLM production-stack router (round-robin and session-ID routing; prefix-aware routing is listed as work in progress, checked 2026-10-23 on the project's GitHub page), SGLang's router, NVIDIA Dynamo's KV-aware router.

## At work
- **Anthropic's postmortem (2025-09)** found that a routing change sent short requests to servers built for a long context window, and that sticky routing kept follow-ups on the wrong servers. Routing is a quality risk too → lesson 35 (rollouts with eval gates).
- **Cloudflare (2026-05)** tracks the estimated in-flight prefill and decode tokens of each endpoint, so a long prompt counts for more than a short one. Counting requests is not enough → lesson 30 (routing).
- **LiteLLM's proxy** limits tokens per minute per key and team, not only requests per minute, because one request can be 100 or 100k tokens. Its multi-instance counters need Redis → lesson 36 (tenancy).

## Common mistakes
**1. Calling the gateway without its key.** The request has no `Authorization` header. The proxy answers with HTTP 500 "Internal Server Error", and its log says "No api key passed in". It is not a clean 401 in this version, so read the status and the log together. **This cell fails on purpose.**

In [14]:
r = httpx.post(GATEWAY, json={"model": "brightside-qwen", "max_tokens": 4,
               "messages": [{"role": "user", "content": "refund policy?"}]}, timeout=60)
try:
    r.raise_for_status()                                   # the gateway refuses: no key
except httpx.HTTPStatusError as err:
    import re
    time.sleep(1)                                          # give the proxy a moment to write its log
    line = next((l for l in Path("../data/29/proxy.log").read_text().splitlines() if "No api key passed in" in l), "")
    line = re.sub(r"^\d{2}:\d{2}:\d{2} - ", "", line.strip())   # drop the clock time: the rest is the same on every run
    NO_KEY.update({"status": str(err).splitlines()[0], "log": line})
    raise

⏭  Skipped: this cell needs [slow]. Run it yourself, or re-run with TUT_ENABLE=slow.


The status line and the matching proxy log line, from a live run or from the recording. Read them together.

In [15]:
import re
if NO_KEY: print("source: this live run")
else:
    NO_KEY.update(REC["no_key"]); print("source: recorded output")
NO_KEY["log"] = re.sub(r"^\d{2}:\d{2}:\d{2} - ", "", NO_KEY["log"])   # the same clean line from either source
print(NO_KEY["status"]); print(NO_KEY["log"])

source: recorded output


Server error '500 Internal Server Error' for url 'http://127.0.0.1:8290/v1/chat/completions'
LiteLLM Proxy:ERROR: auth_exception_handler.py:197 - litellm.proxy.proxy_server.user_api_key_auth(): Exception occured - No api key passed in.


**The fix:** send the key on every call, as `headers=AUTH` does in `load`. In an app, the key comes from its configuration, not from the code. The status code is not the whole story: a 500 from a gateway can be a bad key, a bad config or a dead replica, so check `proxy.log` before you blame the replicas.

## Production checklist
- One URL and one model name for every replica group. Apps never hold a replica list.
- Route on live load (least busy, or a KV-aware policy), and watch the served counts per replica, not only the client's error rate.
- Set timeouts, a small retry count and a cooldown. Retries add load to a replica that is already failing.
- Run at least two gateway instances behind a health check. Keep the key in `.env`, never in a config file you commit.
- Measure the latency the gateway adds per request, and count it in your SLO.

## Try it
For each run, compute the share of its successful requests that replica c answered. Which run sends c the smaller share, and what does that do to the p95 wait?

In [16]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [17]:
for k, r in RESULTS.items():
    c = r["served"]["c"]
    print(f"{k:<20} c answered {c} of {r['ok']} successful requests: {c / r['ok']:.0%}")
print("the smaller share is the gateway run: least-busy stops choosing the replica that already has work in flight")

without: apps pick   c answered 42 of 80 successful requests: 52%
with: one gateway    c answered 40 of 120 successful requests: 33%
the smaller share is the gateway run: least-busy stops choosing the replica that already has work in flight


Stop the replicas and the gateway. After a live run, the numbers and the 500 status and log line are saved in `data/29/results.json`, so they can become the recording.

In [18]:
for p in PROCS: p.terminate()
if PROCS: (Path("../data/29") / "results.json").write_text(json.dumps({"results": RESULTS, "no_key": NO_KEY}, indent=1))

⏭  Skipped: this cell needs [slow]. Run it yourself, or re-run with TUT_ENABLE=slow.


## Recap
- Apps that pick replicas themselves cannot see each other's load: a random rule sends a dead replica a third of the traffic.
- A gateway is one URL and one model name in front of the replicas. It routes on the load of all the traffic, retries failed calls and cools down a bad replica.
- The gain: 40 of 120 requests failed without the gateway and none with it, and the slow replica's share of successes fell from about half to about a third. Each replica's own queue is still there.
- The gateway does not know prefix caches and is a component to make redundant: lessons 30 and 33.
- **Interview:** "Why put a gateway in front of model replicas?" Strong answer: one endpoint and one list, routing on live load across all traffic, retries and cooldown on failure, and the cost of one more hop and one more component to make redundant.